# Task A — five-fold OOF comparison of the best and second-best ensembles

This diagnostic notebook combines the original Task A training rows with the released labelled validation rows, then evaluates both known ensemble recipes with the same five stratified folds. Every row is scored by a model that did not train on that row. It reports the individual SVM/MuRIL scores and the two locked ensemble scores; it does not train a final model or produce a submission ZIP.

Run 11 uses one reinitialized MuRIL layer with its previously fitted 0.62 SVM / 0.38 MuRIL blend and 0.48 threshold. Run 9 uses two reinitialized layers with its locked 0.57 / 0.43 blend and 0.50 threshold. The weights are locked before this run, so the comparison is not tuned on the same OOF predictions. Expected runtime is roughly 6–10 hours on a Kaggle T4 ×2.

In [ ]:
import os, pathlib, shutil, subprocess, sys, json
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only", "origin", "task-b"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1", "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
from urllib.request import urlretrieve
raw_base = "https://raw.githubusercontent.com/robinpnalex/Hastika-ICON2026/task-b/"
for rel in ["data/raw/hastika_binary_validation.csv", "data/raw/hastika_binary_test.csv"]:
    p = pathlib.Path(rel)
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        print("downloading missing released data:", rel)
        urlretrieve(raw_base + rel, p)
    assert p.exists() and p.stat().st_size > 0, f"missing required data: {rel}"
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, accuracy_score
from hastika.common.preprocessing import dedupe_index
assert torch.cuda.is_available(), "enable a GPU"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Stage the combined labelled corpus

In [ ]:
from pathlib import Path
train_path = Path("data/raw/binary_train.csv")
val_path = Path("data/raw/hastika_binary_validation.csv")
combined = pd.concat([pd.read_csv(train_path), pd.read_csv(val_path)], ignore_index=True)
combined = combined.drop_duplicates("id", keep="first").reset_index(drop=True)
assert len(combined) == 7252 and combined["id"].is_unique
combined_path = Path("data/derived/task_a_oof_train_plus_validation.csv")
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(combined_path, index=False)
CANON_TRAIN = Path("data/raw/binary_train.csv")
BACKUP_TRAIN = Path("data/raw/binary_train.csv.before_oof_compare")
assert not BACKUP_TRAIN.exists(), "stale backup exists; stop and inspect before rerunning"
shutil.copy2(CANON_TRAIN, BACKUP_TRAIN)
shutil.copy2(combined_path, CANON_TRAIN)
print("combined labelled rows:", len(combined))


## 2. Generate matched OOF probabilities

In [ ]:
SVM_TAG = "a_oof_trainval_svm_compare"
RUN11_TAG = "a_oof_trainval_reinit1"
RUN9_TAG = "a_oof_trainval_reinit2"
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm", "--task", "a", "--tag", SVM_TAG, "--demojize"], log=f"artifacts/logs/{SVM_TAG}.log")
common = [sys.executable, "-u", "-m", "hastika.models.muril", "--model", "google/muril-base-cased", "--folds", "5", "--seeds", "42", "--epochs", "6", "--bs", "8", "--grad-accum", "2", "--eval-bs", "32", "--select", "last"]
run(common + ["--tag", RUN11_TAG, "--reinit-layers", "1"], log=f"artifacts/logs/{RUN11_TAG}.log")
run(common + ["--tag", RUN9_TAG, "--reinit-layers", "2"], log=f"artifacts/logs/{RUN9_TAG}.log")
print("all three OOF runs completed")


## 3. Report individual and locked ensemble performance

In [ ]:
keep = dedupe_index(combined["Comment"].tolist(), combined["Label"].tolist(), "task A")
fit_df = combined.iloc[keep].reset_index(drop=True)
y = (fit_df["Label"] == "Hate").astype(int).to_numpy()
def load_oof(tag):
    p = np.load(Path("artifacts/runs") / tag / "oof_probs.npy")
    assert p.shape == (len(y), 2), (tag, p.shape, len(y))
    return p[:, 1]
svm = load_oof(SVM_TAG)
run11 = load_oof(RUN11_TAG)
run9 = load_oof(RUN9_TAG)
rows = []
def add(name, p):
    for threshold_name, threshold in [("argmax", 0.5)]:
        pred = (p > threshold).astype(int)
        rows.append({"model": name, "threshold": threshold_name, "macro_f1": f1_score(y, pred, average="macro"), "accuracy": accuracy_score(y, pred)})
add("demojized TF-IDF/SVM", svm)
add("Run 11 MuRIL, one reinit layer", run11)
add("Run 9 MuRIL, two reinit layers", run9)
add("Run 11 locked ensemble", 0.62 * svm + 0.38 * run11)
rows[-1]["threshold"] = 0.48
rows[-1]["macro_f1"] = f1_score(y, (0.62 * svm + 0.38 * run11 > 0.48).astype(int), average="macro")
rows[-1]["accuracy"] = accuracy_score(y, (0.62 * svm + 0.38 * run11 > 0.48).astype(int))
add("Run 9 locked ensemble", 0.57 * svm + 0.43 * run9)
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
OUT = Path("/kaggle/working/task_a_oof_compare_outputs")
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / "summary.csv", index=False)
np.savez(OUT / "oof_probabilities.npz", y=y, svm=svm, run11=run11, run9=run9)


## 4. Restore the canonical training file

In [ ]:
shutil.move(str(BACKUP_TRAIN), str(CANON_TRAIN))
print("restored", CANON_TRAIN)
print("DOWNLOAD RESULTS FROM:", OUT)
